# 10 — Motor de atendimento
**Hackathon Databricks · Terra Signal · Grupo 4**

**Equipe:** André Ferreira, Antonio Menezes, Júlia Vale e Lucas Menghi

**O que este notebook faz:** leva o modelo para a ligação. Quando a central de atendimento recebe um cliente, a
operadora vê na hora a nota de risco dele, **por que** ele está em risco e **qual oferta** mais reduz esse risco. Para
quem ainda não é cliente, compara pacotes e mostra qual oferecer. O notebook prepara e confere o "motor" que o
aplicativo usa na página **Atendimento**.

**A ideia em uma frase:** o modelo do 06 já sabe calcular a nota de qualquer ficha de cliente. Se a operadora muda a
ficha ("e se ele passar para o contrato anual?"), o motor recalcula a nota, e a diferença mostra quanto a oferta ajuda.

| # | Pergunta de negócio | Etapa |
|---|---|---|
| A1 | O motor do aplicativo calcula exatamente a mesma nota do modelo oficial? | 1 e 2. Exportar e conferir |
| A2 | Por que **este** cliente está em risco? | 3.1 Motivos do cliente |
| A3 | Que oferta mais reduz o risco de cada cliente? | 3.2 e 3.3 Ofertas na carteira |
| A4 | Para quem ainda não é cliente, que pacote oferecer? | 4. Comparação de pacotes |

| Entrada | Notebook | O que traz |
|---|---|---|
| Modelo no MLflow (`models:/m-2dd8…`) | 06 | O pipeline oficial: Lasso com C = 0,3, limiar 0,5046 |
| `gold_churn_treino`, `gold_churn_inferencia`, `silver_*` | 02 e 04 | As fichas dos clientes e a base para conferir as variáveis |
| `gold_churn_inferencia_scored`, `gold_churn_variaveis_modelo` | 05 e 06 | As notas publicadas e as 41 colunas do modelo |
| `gold_churn_segmentos` | 07 | Segmento, quadrante e ação de cada cliente |

| Saída | Para quê |
|---|---|
| `app_monitoramento/dados/modelo_motor.json` | Os parâmetros do modelo e a tabela de preços, que o app usa para calcular a nota |
| `app_monitoramento/dados/clientes_atendimento.csv` | A ficha de cada cliente da carteira, com nota, segmento e oferta principal |
| `gold_churn_ofertas` | A oferta principal e as alternativas de cada cliente da fila |
| `app_monitoramento/dados/relatos_inferencia.csv` | O relato de cada cliente: saída declarada, tom, motivos, abordagem e receptividade (5.3) |

In [ ]:
%run ./_comum

## Glossário

| Termo | Por extenso | O que é | Onde aparece |
|---|---|---|---|
| **Ficha** | — | Os dados de um cliente como estão no cadastro (contrato, internet, pagamento, serviços, tempo de casa) | Todo o notebook |
| **Nota de risco** | Probabilidade de churn do modelo | De 0 a 1; acima do limiar (0,5046), o cliente vira "Yes". É nota para ordenar, não chance exata (balanceamento, 06) | Seções 1 a 4 |
| **Decil** | — | A posição do cliente na carteira: D1 = os 10% de maior risco | Seção 3 |
| **Efeito de um tema** | Contribuição para a nota | Soma de coeficiente × variável padronizada das colunas daquele tema: quanto ele empurra a nota em relação a um cliente médio | 3.1 |
| **Oferta** | — | Uma mudança na ficha que a operadora pode propor (contrato, pagamento, suporte, segurança) | 3.2 |
| **Cortesia** | — | O serviço entra sem mudar a mensalidade, por alguns meses | 3.2 |
| **Tabela de preço** | Regressão linear da mensalidade | Quanto cada item do pacote soma na mensalidade, aprendido no treino | 1.2 e 4 |
| **MLflow** | — | Onde o pipeline do modelo está registrado (06) | 1.1 |
| **D25** | Decisão do checklist | Motor de atendimento (este notebook) | Checklist |

## Checklist de decisões

**A mesma tabela em todos os notebooks.** ☑ = decidido (com a data); ◐ = sugestão nossa, a confirmar (as de variáveis, na modelagem); ☐ = em aberto.

| # | Decisão | Opções | Proposta | Impacto | Evidência no 01 | No 02 | Depende de | Decisão do grupo |
|---|---|---|---|---|---|---|---|---|
| D1 | Grafia de `PhoneService` (70 + 23) | Padronizar | Padronizar para `Yes`/`No` | Sem isso, `yes` e `Yes` viram categorias diferentes no modelo | 4.1 | Aplicado | Carlos: só erro de padronização (respondido) | ☑ Padronizar para `Yes`/`No` (10/11) |
| D2 | `TotalCharges` nulo (278 + 64) | a) preencher com `MonthlyCharges` × `tenure`; b) excluir; c) deixar ausente | a | Excluir não serve para os 64 da inferência, que precisam de previsão, e perde o sinal da 4.3. A regra vale para 99% dos clientes | 4.1, 4.2, 4.3 | Aplicado | G2; Carlos: preencher assim é "razoável, desde que lembrem que é uma aproximação" (respondido); falta saber por que o valor está ausente | ☑ Preencher com `MonthlyCharges` × `tenure`, como aproximação (10/11) |
| D3 | `TotalCharges` nulo com `tenure` = 0 (10 + 1) | a) 0; b) outro valor | a (sai da própria regra: mensalidade × 0) | Clientes novos, sem cobrança ainda; efeito pequeno | 4.2 | Aplicado | G2 | ☑ 0 (10/11) |
| D4 | `tenure` = `unknown` (19 + 4) | a) `TotalCharges` ÷ `MonthlyCharges`, arredondado; b) mediana; c) excluir | a; mediana do histórico no único cliente sem as duas colunas | `tenure` é das variáveis mais ligadas ao churn: um erro aqui muda a faixa de tempo de casa do cliente | 4.1, 4.2 | Aplicado | G2; Carlos: 72 meses é teto real (respondido) | ☑ `TotalCharges` ÷ `MonthlyCharges`, arredondado e limitado a 72; mediana do histórico no caso sem os dois (10/11) |
| D5 | Indicador `fl_total_charges_ausente` | Criar ou não | Criar e comparar o modelo com e sem ele | Preserva o sinal da ausência (33,1% contra 26,4% de churn, p = 0,016) | 4.3 | Criado | Carlos: por que falta `TotalCharges` | ☑ Criar; o modelo testa com e sem (10/11) |
| D6 | Indicador `fl_tenure_ausente` | Criar ou não | Criar, mas tende a sair na seleção (só 19 casos, p = 0,21) | Pouca informação; uma coluna a mais | 4.3 | Não criado | G2 | ☑ Não criar: 19 casos, p = 0,21 (10/11) |
| D7 | `TotalCharges` fora de ±20% da regra (47 + 9) | a) manter; b) investigar; c) excluir | a | Mostra que o preenchimento de D2 é uma aproximação | 4.2 | Mantidos | Carlos: o preço muda com plano e promoções, e não há histórico de preços (respondido) | ☑ Manter (10/11) |
| D8 | Quebras de linha no `CustomerFeedback` (85 + 15) | Remover ou manter | Remover, se o texto for usado | Só importa se o texto entrar na análise ou no modelo | 4.2 | Aplicado | D9 | ☑ Trocar a quebra de linha por espaço (10/11) |
| D9 | `CustomerFeedback` no modelo | a) com; b) sem; c) os dois | c, com o modelo sem texto como principal | Possível vazamento: uma regra sobre o texto acerta 98,9% | Célula 2 (seção 1); 7.3 do 03 | Fora do 02 (modelo) | G1; Carlos: feedback antes ou depois do cancelamento | ☑ Fora da ABT. O texto vai na tabela separada `gold_churn_texto`, para o modelo com texto (contraexemplo); o modelo oficial é sem texto (10/11) |
| D10 | Serviços com 3 categorias ou sim/não | a) manter 3; b) binário | Testar as duas | "No internet service" repete `InternetService` em 6 colunas: redundância | 3 | Fora do 02 (notebook 04) | G4 | ◐ Sugestão: serviços como sim/não (`_bin`), sempre junto com `InternetService`. A ABT traz as duas versões; seleção final na modelagem (10/11) |
| D11 | Fonte e schema do grupo | a) CSVs de `dados/`; b) tabelas `bronze_*` em `terra_signal` | b | Todos os notebooks; os números precisam bater entre os integrantes | 1.2, 2 | Aplicado: o 02 lê das tabelas `bronze_*` | G3; Carlos: CSV com quebra de linha, schema | ☑ Tabelas `bronze_*` em `workspace.terra_signal`; ignorar (sem apagar) as do upload em `workspace.default` (10/11) |
| D12 | `MonthlyIncome` no modelo | a) manter como candidata; b) ignorar | a, e deixar a seleção decidir | Base externa, sem garantia de precisão; os organizadores dizem que pode ser ignorada. Na exploratória, a renda não muda o risco | 3; seção 10 do 03 | Convertida para número e mantida | Carlos: renda familiar em R$, base externa (respondido) | ◐ Sugestão: descartar (IV abaixo de 0,02, base externa). Fica na ABT; seleção final na modelagem (10/11) |
| D13 | Linhas do `Prediction.csv` | a) 1.409 (clientes); b) 1.461 (linhas físicas) | a | Arquivo com o número errado de linhas pode invalidar a entrega | 1.2 | Fora do 02 (entrega) | Organização: confirmar | ☑ 1.409, uma por `customerID`; a confirmar com a organização (10/11) |
| D14 | Formato da gold | — | `gold_churn_treino` (5.634, com `Churn` 1/0) e `gold_churn_inferencia` (1.409, sem `Churn`), mesmas colunas e ordem; números como número, categorias como texto; comentário em cada coluna; `gold_churn_previsoes` depois do modelo | É o que a modelagem usa; formato estável evita retrabalho | — | Fora do 02 (notebook 04) | — | ☑ ABT com todas as candidatas (originais limpas, serviços nas duas versões, indicador e derivadas) + dicionário com a sugestão de cada coluna; texto em `gold_churn_texto` (10/11) |
| D15 | Outras variáveis sem poder preditivo | a) descartar já; b) deixar a seleção decidir | a: `gender`, `PhoneService` e `MultipleLines` | IV abaixo de 0,02; modelo mais simples de explicar; sexo é variável sensível | Seção 7 e 8 do 03 | Fora do 02 (notebook 04) | — | ◐ Sugestão: descartar `gender`, `PhoneService` e `MultipleLines` (IV abaixo de 0,02). Ficam na ABT; seleção final na modelagem (10/11) |
| D16 | `MonthlyCharges` e `TotalCharges` (repetem outras variáveis) | a) manter; b) trocar por derivada; c) deixar a seleção decidir | Decidir no 04, com as derivadas | Repetem `InternetService` e `tenure` (associação ≥ 0,8) | Seção 6 e 8 do 03 | Fora do 02 (notebook 04) | 04 | ◐ Sugestão: manter `MonthlyCharges` (ou a mensalidade relativa ao pacote) e descartar `TotalCharges` (repete `tenure`, ρ 0,89). As três ficam na ABT; seleção final na modelagem (10/11) |
| D17 | Fonte do 03 | a) CSVs; b) tabelas `bronze_*` | b | Coerência com a D11; os números não mudam | — | Fora do 02 (notebook 03) | — | ☑ Ler da bronze (10/11) |
| D18 | Divisão de papéis | — | A preparação entrega a ABT e sugere; a seleção final das variáveis é feita na modelagem | Quem decide a seleção final das variáveis | — | — | — | ☑ Como proposto (10/11) |
| D19 | Seleção de variáveis | a) só as recomendadas; b) recomendadas + seleção para frente por AUC; c) testes estatísticos com correção para múltiplos testes e retirada das duplicatas | c, conferida com b | É o que entra no modelo | — | — | Notebook 05 | ☑ As 41 colunas do notebook 05 (seções 2 e 3): saem 6 categóricas redundantes e 12 variáveis (duplicatas exatas, versões repetidas do tempo de casa, `gender` sem associação após a correção FDR). A conferência com os métodos da aula (11 variáveis, AUC 0,841) cabe dentro das 41 (11/11) |
| D20 | Segmentação de clientes | a) sem segmentação; b) K-means no comportamento, sem o churn, cruzado com risco × valor | b | Diz que tipo de cliente está em risco e que oferta faz sentido | — | — | Notebook 07 | ◐ Sugestão para o negócio e a apresentação (11/11) |
| D21 | Modelo final | a) logística simples; b) regularização (Lasso, Ridge, Elastic Net); c) árvores (*random forest*, XGBoost, CatBoost) | O de melhor desempenho e mais consistente entre treino, teste e base total | É o modelo que vai para a entrega | — | — | Notebook 06 | ☑ Notebook 06 (modelo do grupo): logística Lasso, 41 colunas, C = 0,3, classes balanceadas, vencedora pela regra de consistência (Ridge a 0,003); AUC 0,830 no teste. Atenção: o balanceamento infla a probabilidade (média de 41% para 27% de churn real) (11/11) |
| D22 | Ponto de corte do Sim/Não | a) 0,5; b) o que maximiza o F1; c) pelo custo da ação | b | Define quantos clientes viram "Yes" | — | — | Notebook 06 | ☑ 0,5046: maior F1 nas probabilidades fora da dobra do treino (notebook 06) (11/11) |
| D23 | Arquivo da entrega | — | 1.409 linhas, `customerID` e `prediction` (Yes/No), na ordem do `inference.csv`, no formato do notebook oficial | Arquivo fora do formato pode invalidar a entrega | — | — | D13; notebook 08 | ☑ `prediction_grupo4_<data_hora>.csv` gerado no 08 e conferido nas 8 regras: 571 de 1.409 com Yes (40,5%); falta comparar os nomes das colunas com o notebook oficial (11/11) |
| D24 | Monitoramento do modelo | a) sem monitoramento; b) PSI das variáveis e das notas, taxa de Yes e reprodução pelo MLflow, com semáforo e app | b | Avisa quando o modelo envelhece, antes de o resultado de negócio mostrar | — | — | Notebook 09 | ◐ Proposta: notebook 09 e app em Streamlit (11/11) |
| D25 | Motor de atendimento | a) só a fila de retenção; b) a nota, os motivos e a oferta de cada cliente na hora da ligação, e a comparação de pacotes para quem ainda não é cliente | b | Leva o modelo para quem fala com o cliente | — | — | Notebook 10 | ◐ Proposta: notebook 10 e página Atendimento do app (12/11) |

## 1. Exportar o modelo para o aplicativo

> **A1 — O motor do aplicativo calcula exatamente a mesma nota do modelo oficial?**

**Teoria: por que exportar os parâmetros, e não o modelo inteiro.** O modelo é uma regressão logística: a nota é

> **nota = 1 ÷ (1 + e^−(intercepto + Σ coeficiente × variável padronizada))**

Para calcular essa conta, bastam quatro listas: a mediana de cada coluna (para preencher ausentes), a média e o
desvio (para padronizar) e os coeficientes. São 41 números de cada. Guardando só isso num arquivo, o aplicativo
calcula a nota sem precisar do MLflow, do scikit-learn ou de um *cluster*, e abre na hora. A mesma conta também mostra
**quanto cada variável pesou** para o cliente (coeficiente × variável padronizada), que é o "porquê" da nota.

**Passo a passo:**
1. Carrego o pipeline oficial do MLflow (se não carregar, refaço o Lasso com C = 0,3 na mesma separação do 05: é
   determinístico e a 2.1 confere que dá a mesma nota).
2. Tiro dele a mediana, a média, o desvio, os coeficientes e o intercepto.
3. Junto as médias de mensalidade por tipo de internet (usadas na `mensalidade_relativa_pacote`), os cortes dos decis
   da carteira e a tabela de preço.

In [ ]:
# 1.1 Carrega o modelo oficial (MLflow) e as bases
import sys, json
PASTA_APP = os.path.join(raiz, "app_monitoramento")
if PASTA_APP not in sys.path:
    sys.path.insert(0, PASTA_APP)                              # o mesmo código que o aplicativo usa
from motor import Motor
from motor.abt import montar_abt
from motor.modelo import ACOES, NOMES_CONTRATO, NOMES_PAGAMENTO, novo_cliente
from motor.exportar import parametros_do_modelo, tabela_de_preco

assert "FQ" in globals(), "rode antes a célula %run ./_comum"
treino = spark.table(f"{FQ}.gold_churn_treino").toPandas().sort_values("customerID").reset_index(drop=True)
inferencia = spark.table(f"{FQ}.gold_churn_inferencia").toPandas()
inf_scored = spark.table(f"{FQ}.gold_churn_inferencia_scored").toPandas()
seg = spark.table(f"{FQ}.gold_churn_segmentos").toPandas()
info = spark.table(f"{FQ}.gold_churn_modelo_info").toPandas().iloc[0]
var = spark.table(f"{FQ}.gold_churn_variaveis_modelo").toPandas().sort_values(["tipo", "ordem"])
LISTAS = {t: g["variavel"].tolist() for t, g in var.groupby("tipo")}
COLUNAS, CATEGORICAS = LISTAS["modelo"], LISTAS["categorica_dummy"]

# endereço do registro gravado pelo 06 (6.4) no resumo do modelo; o ID fixo é o da execução de 11/11
MODEL_URI = info.get("model_uri") if isinstance(info.get("model_uri"), str) else "models:/m-2dd8f256aa384a98bf46d77f84ff80eb"
try:
    import mlflow
    pipeline = mlflow.sklearn.load_model(MODEL_URI)
    ORIGEM = f"MLflow ({MODEL_URI})"
except Exception as erro:
    print("MLflow não carregou:", type(erro).__name__, str(erro)[:150], "→ refaço o Lasso na separação do 05")
    from sklearn.pipeline import Pipeline
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler, FunctionTransformer
    from sklearn.linear_model import LogisticRegression
    def preparar_base(df):                                        # a mesma limpeza do 06 (6.1)
        df = pd.get_dummies(df.drop(columns=LISTAS["redundante"], errors="ignore"), columns=CATEGORICAS, drop_first=True, dtype=float)
        X = df.select_dtypes(include=["number", "bool"]).drop(columns=["customerID", "Churn"], errors="ignore")
        X = X.drop(columns=LISTAS["retirada"], errors="ignore").apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
        return X.reindex(columns=COLUNAS, fill_value=0.0).astype(float)
    split = spark.table(f"{FQ}.gold_churn_split").toPandas()
    tr = treino.set_index("customerID").loc[split.loc[split["base"] == "Treino"].sort_values("ordem")["customerID"]].reset_index()
    lasso = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()),
                      ("modelo", LogisticRegression(penalty="l1", solver="saga", max_iter=10000, class_weight="balanced",
                                                    random_state=42, C=0.3))]).fit(preparar_base(tr), tr["Churn"])
    pipeline = Pipeline([("limpeza", FunctionTransformer(preparar_base)), ("modelo", lasso)])
    ORIGEM = "Lasso refeito na separação do 05 (C = 0,3)"

print("Modelo:", ORIGEM)
print(f"Treino: {virg(len(treino), 0)} · carteira: {virg(len(inferencia), 0)} · colunas do modelo: {len(COLUNAS)}")

In [ ]:
# 1.2 Parâmetros do motor e tabela de preço
PARAMETROS = parametros_do_modelo(pipeline, COLUNAS, CATEGORICAS, info["limiar"], inf_scored["Probabilidade_Churn"], treino)
PARAMETROS["preco"], R2_PRECO = tabela_de_preco(treino)
motor = Motor(PARAMETROS)

coefs = pd.Series(PARAMETROS["coeficientes"], index=COLUNAS)
print(f"Coeficientes: {len(coefs)} (zerados pela Lasso: {int((coefs == 0).sum())}) · intercepto {virg(PARAMETROS['intercepto'], 3)} · limiar {virg(PARAMETROS['limiar'], 4)}")
tab_preco = pd.Series(PARAMETROS["preco"]).rename("R$ por mês").round(2)
display(tab_preco.to_frame())
print(f"A tabela de preço explica {virg(100 * R2_PRECO, 1)}% da variação da mensalidade no treino")

> **Como ler (1.2)**
> - **Coeficientes:** são os mesmos 41 do 06; os zerados (11) são as colunas que a Lasso descartou.
> - **Tabela de preço:** cada linha é quanto o item soma na mensalidade. Uma regressão linear da mensalidade pelos
>   itens do pacote explica quase toda a variação: a Terra Signal tem, na prática, uma tabela de preços fixa, e os
>   dados a revelam (telefone cerca de R$ 20, DSL R$ 25, fibra R$ 50, cada proteção R$ 5, cada streaming R$ 10).
> - **Para que serve:** quando a operadora monta um pacote para quem ainda não é cliente, o app estima a mensalidade
>   por essa tabela e calcula a nota com ela.

## 2. Conferir o motor

**O que faço:** o motor só pode ir para o aplicativo se devolver **exatamente** o que o 04 e o 06 devolvem. Confiro
cinco regras; se alguma falhar, o notebook para.

In [ ]:
# 2.1 O motor reproduz a gold e as notas publicadas?
regras = []
for base, silver in [("treino", "silver_history"), ("inferencia", "silver_inference")]:
    gold = treino if base == "treino" else inferencia
    abt = montar_abt(spark.table(f"{FQ}.{silver}").toPandas(), PARAMETROS["media_pacote"])
    abt = abt.set_index("customerID").loc[gold["customerID"]].reset_index()[gold.columns]
    num = gold.select_dtypes("number").columns
    regras.append((f"As variáveis do motor = gold_churn_{base} ({len(gold.columns)} colunas)",
                   bool(np.allclose(abt[num].to_numpy(float), gold[num].to_numpy(float), equal_nan=True)
                        and (abt.drop(columns=num).astype(str) == gold.drop(columns=num).astype(str)).all().all())))
p_motor = motor.probabilidade(inf_scored)
p_pipeline = pipeline.predict_proba(inferencia.set_index("customerID").loc[inf_scored["customerID"]].reset_index())[:, 1]
regras += [
    ("Nota do motor = nota publicada (gold_churn_inferencia_scored)", bool(np.allclose(p_motor, inf_scored["Probabilidade_Churn"]))),
    ("Nota do motor = nota do pipeline oficial", bool(np.allclose(p_motor, p_pipeline))),
    ("Decil do motor = decil publicado", bool((pd.Series([f"D{motor.decil(p)}" for p in p_motor]) == inf_scored["Decil_Risco"].to_numpy()).all())),
    ("Tabela de preço explica pelo menos 99% da mensalidade", R2_PRECO >= 0.99),
]
conf = pd.DataFrame(regras, columns=["regra", "ok"])
display(conf)
print(f"Maior diferença de nota: {np.abs(p_motor - inf_scored['Probabilidade_Churn']).max():.1e}")
assert conf["ok"].all(), "o motor não reproduz o modelo oficial: não publicar o app"
RESPOSTAS = {"A1": f"Sim: as {len(regras)} conferências deram True (diferença máxima de nota de {np.abs(p_motor - inf_scored['Probabilidade_Churn']).max():.0e})"}
print("Resposta A1:", RESPOSTAS["A1"])

> **Como ler (2.1)**
> - **As duas primeiras regras** garantem que o aplicativo monta as variáveis derivadas igual ao 04: é o que permite
>   mudar a ficha (por exemplo, o contrato) e recalcular tudo o que depende dela (`IsHighRiskContract`,
>   `new_monthly_customer`...).
> - **As duas do meio** garantem que a nota é a do modelo oficial, até a 16ª casa decimal.
> - **Se alguma der False:** o 04 ou o 06 mudou depois que o motor foi escrito. Ajustar `motor/abt.py` antes de publicar.

## 3. O que a operadora vê: motivos e ofertas

> **A2 — Por que este cliente está em risco?** · **A3 — Que oferta mais reduz o risco de cada cliente?**

**Teoria: o porquê da nota.** Na logística, a nota é a soma de pedaços: cada coluna contribui com
coeficiente × valor padronizado. Somando as colunas de um mesmo tema (tempo de casa, contrato, pagamento, suporte,
pacote, perfil), sai quanto cada tema empurra a nota **deste** cliente para cima ou para baixo, em relação a um cliente
médio. Juntar por tema também resolve a leitura das variáveis que andam juntas (preço e tipo de internet, 06 seção 5).

**Teoria: a oferta.** Para cada ação que a operadora pode propor, mudo a ficha, recalculo a nota e meço quanto ela cai:

| Ação | O que muda na ficha | Vale para |
|---|---|---|
| Contrato anual | `Contract` = One year | Clientes no mensal |
| Contrato de dois anos | `Contract` = Two year | Mensal ou anual |
| Débito automático | `PaymentMethod` = Bank transfer (automatic) | Cheque eletrônico ou boleto |
| Suporte técnico | `TechSupport` = Yes (cortesia) | Com internet e sem suporte |
| Segurança online | `OnlineSecurity` = Yes (cortesia) | Com internet e sem segurança |

Testo cada ação sozinha e cada par (sem dois contratos ao mesmo tempo).

> **Cuidado importante:** a queda da nota é o que o **modelo** associa à mudança, não a garantia de que o cliente fica.
> Quem tem contrato de dois anos é, em parte, quem já era fiel. Por isso as ofertas são uma ordem de tentativa para a
> operadora e uma hipótese a testar (oferecer a metade da fila e comparar com a outra metade).

In [ ]:
# 3.1 Um cliente da fila: nota, motivos e ofertas
fila = inf_scored.merge(seg.loc[seg["base"] == "inferencia", ["customerID", "segmento", "nome_segmento", "quadrante"]], on="customerID")
exemplo = fila[(fila["quadrante"] == "Salvar já") & (fila["segmento"] == "S1")].sort_values("Probabilidade_Churn", ascending=False).iloc[0]
cliente = exemplo.to_dict()
p0 = motor.prob1(cliente)
print(f"Cliente {cliente['customerID']} · {cliente['nome_segmento']} · nota {virg(p0, 2)} · decil D{motor.decil(p0)} · "
      f"mensalidade R$ {virg(cliente['MonthlyCharges'], 2)}")
display(motor.motivos(cliente).round(3))
of = motor.ofertas(cliente)
display(of[["oferta", "nota_antes", "nota_depois", "decil_depois"]].head(6).round(3))
top = motor.motivos(cliente).index[:3].tolist()
RESPOSTAS["A2"] = f"No exemplo {cliente['customerID']}: " + "; ".join(f"{t.lower()} ({motor.motivos(cliente).loc[t, 'descricao']})" for t in top)
print("Resposta A2:", RESPOSTAS["A2"])

> **Como ler (3.1)**
> - **Motivos:** a coluna `efeito` é quanto o tema soma na nota (na escala do modelo, antes de virar probabilidade).
>   Positivo = aumenta o risco; negativo = protege. A `descricao` é o que a operadora lê na tela, com os dados do
>   próprio cliente.
> - **Ofertas:** a nota antes e depois de cada oferta, da que mais reduz para a que menos reduz. Na tela, a operadora
>   vê a principal e as alternativas, caso o cliente recuse a primeira.

In [ ]:
# 3.2 As ofertas na fila inteira: quantos clientes saem da zona de risco com cada uma
fila_yes = fila[fila["Churn_Previsto"] == 1].reset_index(drop=True)
todas = motor.ofertas_em_lote(fila_yes)
todas["sai_da_zona"] = todas["nota_depois"] < motor.limiar
efeito = (todas.groupby("oferta").agg(clientes_validos=("customerID", "size"), queda_media_da_nota=("reducao", "mean"),
                                       saem_da_zona=("sai_da_zona", "sum"))
          .sort_values("saem_da_zona", ascending=False))
display(efeito.round(3))

# escada de ofertas de cada cliente: a principal, a melhor sem fidelidade de 2 anos e a melhor sem mexer no contrato
def melhor(df):
    return df.sort_values("nota_depois").groupby("customerID").head(1).set_index("customerID")
tem_contrato = todas["acoes"].apply(lambda a: any(x.startswith("contrato") for x in a))
tem_2anos = todas["acoes"].apply(lambda a: "contrato_2anos" in a)
escada = (melhor(todas)[["oferta", "nota_depois"]].rename(columns={"oferta": "oferta_principal", "nota_depois": "nota_principal"})
          .join(melhor(todas[~tem_2anos])[["oferta", "nota_depois"]].rename(columns={"oferta": "oferta_sem_2anos", "nota_depois": "nota_sem_2anos"}))
          .join(melhor(todas[~tem_contrato])[["oferta", "nota_depois"]].rename(columns={"oferta": "oferta_sem_contrato", "nota_depois": "nota_sem_contrato"})))
n = len(fila_yes)
saem = {k: int((escada[f"nota_{k}"] < motor.limiar).sum()) for k in ["principal", "sem_2anos", "sem_contrato"]}
print(f"Fila: {virg(n, 0)} clientes com Yes. Pela nota do modelo, saem da zona de risco: {saem['principal']} com a oferta principal, "
      f"{saem['sem_2anos']} com a melhor sem fidelidade de 2 anos e {saem['sem_contrato']} sem mexer no contrato")
RESPOSTAS["A3"] = (f"Mudar o contrato é a alavanca mais forte: com a oferta principal, {saem['principal']} dos {n} saem da zona de risco "
                   f"pela nota; sem mexer no contrato, só {saem['sem_contrato']}. A escada de três ofertas de cada cliente vai para a tela")
print("Resposta A3:", RESPOSTAS["A3"])

In [ ]:
# 3.3 Gráfico: quantos clientes da fila saem da zona de risco, por oferta (só ações sozinhas e a escada)
sozinhas = efeito[efeito.index.isin([d["rotulo"] for d in ACOES.values()])]["saem_da_zona"]
barras = pd.concat([sozinhas, pd.Series({"Escada: melhor sem mexer no contrato": saem["sem_contrato"],
                                         "Escada: melhor sem fidelidade de 2 anos": saem["sem_2anos"],
                                         "Escada: oferta principal": saem["principal"]})]).sort_values()
fig, ax = plt.subplots(figsize=(10, 4.6))
yy = np.arange(len(barras))
cores = [DESTAQUE if i.startswith("Escada: oferta") else COR if i.startswith("Escada") else CINZA for i in barras.index]
ax.barh(yy, barras.values, color=cores, height=0.6)
for y0, v in zip(yy, barras.values):
    ax.text(v + n * 0.01, y0, f"{int(v)} de {n} ({virg(100 * v / n, 0)}%)", va="center", fontsize=9, color=TEXTO)
ax.set_yticks(yy, barras.index, fontsize=9.5)
ax.set_xlim(0, n * 1.25); ax.set_xticks([])
ax.spines[["bottom", "left"]].set_visible(False); ax.tick_params(axis="y", length=0)
fig.suptitle(f"Pela nota do modelo, mexer no contrato é o que mais tira clientes da zona de risco: a oferta principal\n"
             f"tira {saem['principal']} dos {n}; sem mudar o contrato, só {saem['sem_contrato']}", x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, ["Contract", "PaymentMethod", "TechSupport", "OnlineSecurity"], fonte="gold_churn_inferencia_scored; motor do notebook 10")
plt.show()

> **Como ler (3.3)**
> - **Onde olhar:** o comprimento de cada barra = quantos dos clientes com "Yes" passam a ficar abaixo do limiar se
>   aceitarem a oferta.
> - **O que comparar:** as ações sozinhas (cinza) com a escada (azul e laranja). Suporte, segurança e débito
>   automático sozinhos mudam pouco a nota; combinados com o contrato, tiram mais clientes da zona de risco.
> - **Achado:** o contrato é a alavanca principal, como no 06 (tempo de casa e contrato somam mais da metade do peso).
>   Na ligação, a operadora começa pela oferta principal e desce a escada se o cliente recusar.
> - **Cuidado:** é o efeito **pela nota do modelo**, não a retenção garantida. A taxa real de aceite e de permanência
>   só um teste com parte da fila mostra.

## 4. Quem ainda não é cliente: que pacote oferecer

> **A4 — Para quem ainda não é cliente, que pacote oferecer?**

**O que faço:** a operadora preenche o perfil (idoso, parceiro, dependentes) e o pacote pedido (internet, telefone,
streaming, forma de pagamento). O motor monta a ficha de um cliente no primeiro mês, estima a mensalidade pela tabela de
preço e compara variações do mesmo pacote: os três contratos, com e sem débito automático, com e sem suporte e
segurança. A recomendação é o pacote de menor nota de risco.

**Limite:** um cliente novo no contrato de dois anos é raro no histórico (quase ninguém começa assim), então a nota
desses pacotes é uma extrapolação. A comparação serve para **ordenar** as opções, não como previsão exata.

In [ ]:
# 4.1 Exemplo: pedido de fibra com telefone e streaming de TV, pagando por cheque eletrônico
perfil = {"idoso": 0, "parceiro": 0, "dependentes": 0, "renda": None}
pedido = {"internet": "Fiber optic", "telefone": 1, "linhas_multiplas": 0, "fatura_digital": 1, "pagamento": "Electronic check",
          "contrato": "Month-to-month", "StreamingTV": 1}
opcoes = []
for contrato in ["Month-to-month", "One year", "Two year"]:
    for pagamento in [pedido["pagamento"], "Bank transfer (automatic)"]:
        for protecao in [False, True]:
            pac = {**pedido, "contrato": contrato, "pagamento": pagamento, "TechSupport": protecao, "OnlineSecurity": protecao}
            c = novo_cliente(perfil, pac, motor)
            p = motor.prob1(c)
            opcoes.append({"contrato": contrato, "pagamento": pagamento, "suporte_e_seguranca": protecao,
                           "mensalidade": round(c["MonthlyCharges"], 2), "nota": p, "decil": motor.decil(p)})
opcoes = pd.DataFrame(opcoes).drop_duplicates(subset=["contrato", "pagamento", "suporte_e_seguranca"]).sort_values("nota")
display(opcoes.round(3))
pedido_nota = opcoes[(opcoes["contrato"] == "Month-to-month") & (opcoes["pagamento"] == pedido["pagamento"]) & ~opcoes["suporte_e_seguranca"]].iloc[0]
rec = opcoes.iloc[0]
RESPOSTAS["A4"] = (f"No exemplo, o pacote pedido (mensal, cheque eletrônico) tem nota {virg(pedido_nota['nota'], 2)}; o recomendado "
                   f"(contrato {NOMES_CONTRATO[rec['contrato']]}, {NOMES_PAGAMENTO[rec['pagamento']]}{', com suporte e segurança' if rec['suporte_e_seguranca'] else ''}) "
                   f"tem nota {virg(rec['nota'], 2)}, por R$ {virg(rec['mensalidade'], 2)}")
print("Resposta A4:", RESPOSTAS["A4"])

> **Como ler (4.1)**
> - **Cada linha é uma variação do pacote pedido**, com a mensalidade estimada e a nota de risco no primeiro mês.
> - **O que observar:** quanto a nota cai ao trocar o contrato e o pagamento, e quanto custa incluir suporte e
>   segurança (R$ 10 por mês na tabela).
> - **Na tela:** a operadora vê o pacote pedido, o recomendado e o argumento ("com o plano anual e débito automático,
>   o cliente fica no grupo de menor risco e paga o mesmo").

## 5. Gravar os arquivos do aplicativo

**O que faço:** gravo o `modelo_motor.json` (os parâmetros) e o `clientes_atendimento.csv` (a ficha de cada cliente da
carteira, com nota, segmento e a escada de ofertas) na pasta do aplicativo, e a tabela `gold_churn_ofertas` no catálogo. A 5.2 grava a curva de ganho do teste e a 5.3, o relato de cada cliente.

In [ ]:
# 5.1 Grava modelo_motor.json, clientes_atendimento.csv e gold_churn_ofertas
os.makedirs(os.path.join(PASTA_APP, "dados"), exist_ok=True)
with open(os.path.join(PASTA_APP, "dados", "modelo_motor.json"), "w", encoding="utf-8") as f:
    json.dump(PARAMETROS, f, ensure_ascii=False, indent=1)

clientes = fila.merge(seg.loc[seg["base"] == "inferencia", ["customerID", "acao_sugerida", "receita_em_risco"]], on="customerID", how="left")
clientes = clientes.merge(escada.reset_index(), on="customerID", how="left")
clientes.to_csv(os.path.join(PASTA_APP, "dados", "clientes_atendimento.csv"), index=False)

ofertas = escada.reset_index().assign(nota_atual=lambda d: d["customerID"].map(fila.set_index("customerID")["Probabilidade_Churn"]))
spark.createDataFrame(ofertas).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.gold_churn_ofertas")
spark.sql(f"COMMENT ON TABLE {FQ}.gold_churn_ofertas IS 'Escada de ofertas de cada cliente com Yes: principal, sem 2 anos e sem contrato, com a nota depois (notebook 10)'")
for nome in ["modelo_motor.json", "clientes_atendimento.csv"]:
    print(os.path.join(PASTA_APP, "dados", nome))
print(f"{FQ}.gold_churn_ofertas: {virg(len(ofertas), 0)} linhas")

In [ ]:
# 5.2 Tabela de ganho do teste: taxa REAL de churn por decil (curva de ganho e simulador de campanha do app)
abt_scored = spark.table(f"{FQ}.gold_churn_abt_scored").toPandas()
teste = abt_scored[abt_scored["Base_Modelagem"] == "Teste"].copy()
teste["decil"] = pd.qcut(teste["Probabilidade_Churn"].rank(method="first", ascending=False), 10, labels=range(1, 11)).astype(int)
ganho = (teste.groupby("decil").agg(clientes=("Churn", "size"), churners=("Churn", "sum"), nota_media=("Probabilidade_Churn", "mean"))
         .reset_index())
ganho["taxa_churn"] = ganho["churners"] / ganho["clientes"]
ganho["pct_carteira_acum"] = ganho["clientes"].cumsum() / ganho["clientes"].sum()
ganho["captura_acum"] = ganho["churners"].cumsum() / ganho["churners"].sum()
ganho.to_csv(os.path.join(PASTA_APP, "dados", "ganho_teste.csv"), index=False)
display(ganho.round(3))
print(f"D1: nota média {virg(ganho.loc[0, 'nota_media'], 2)} para churn real de {virg(100 * ganho.loc[0, 'taxa_churn'], 1)}% · "
      f"20% de maior risco capturam {virg(100 * ganho.loc[1, 'captura_acum'], 1)}% de quem sai (iguais ao 06, seção 4)")

> **Como ler (5.2):** cada linha é um décimo do teste, do maior risco (1) para o menor (10). A `taxa_churn` é o churn
> **real** do grupo; a `nota_media`, o que o modelo deu. A nota fica acima da taxa (efeito do balanceamento, 06), por isso o
> simulador de campanha do app usa a taxa real. A `captura_acum` é a curva de ganho: contatando os 20% de maior risco,
> alcançamos 48% de quem sai.

In [ ]:
# 5.3 O relato de cada cliente da carteira: saída declarada, tom, motivos, abordagem e receptividade (app, Atendimento)
#     O relato NÃO entra na nota (vazamento, D9): serve para escolher a conversa e ordenar a lista do atendimento ativo.
from motor.relatos import montar_relatos
silver_inf = spark.table(f"{FQ}.silver_inference").toPandas()
relatos = montar_relatos(silver_inf, inf_scored.set_index("customerID")["Churn_Previsto"])
relatos.to_csv(os.path.join(PASTA_APP, "dados", "relatos_inferencia.csv"), index=False)

# a saída declarada é a mesma marcação do 04 (gold_churn_texto.fl_menciona_cancelamento)
texto = spark.table(f"{FQ}.gold_churn_texto").toPandas()
flag_04 = texto[texto["base"] == "inferencia"].set_index("customerID")["fl_menciona_cancelamento"].astype(int)
assert (relatos.set_index("customerID")["decidiu_sair"] == flag_04.reindex(relatos["customerID"]).to_numpy()).all(), \
    "a regra de saída declarada difere do notebook 04"

display(pd.crosstab(relatos["abordagem"], relatos["receptividade"], margins=True, margins_name="Total"))
motivos = relatos["motivos_citados"].replace("", None).dropna().str.split("; ").explode().value_counts()
display(motivos.rename("clientes que citam").to_frame())
print(f"Saída declarada no relato: {relatos['decidiu_sair'].sum()} de {len(relatos)} clientes "
      f"({virg(100 * relatos['decidiu_sair'].mean(), 1)}%); igual à marcação do 04")

> **Como ler (5.3)**
> - **Abordagem:** a conversa a ter com cada cliente. **Recuperação** = o relato já diz que ele decidiu sair (a mesma regra
>   do 04, que acerta 98,9% no histórico); **retenção** = o modelo prevê saída e o relato não diz que ele decidiu;
>   **fidelização** = risco baixo.
> - **Receptividade:** a chance de o cliente ouvir uma oferta, pelo tom do relato (alta, média ou baixa). É uma
>   aproximação até existir o histórico de contatos, que o app começa a gravar a cada atendimento registrado.
> - **Motivos citados:** as queixas que aparecem nos relatos negativos (preço, conexão, concorrência, forma de pagamento,
>   contrato, atendimento), lidas por regra de palavras nos trechos sem negação.
> - **O relato não entra na nota:** foi escrito depois da decisão do cliente (D9). No app, ele só escolhe a conversa e a
>   ordem da lista do atendimento ativo.

## Respostas às perguntas de negócio

In [ ]:
PERGUNTAS = {"A1": "O motor do aplicativo calcula exatamente a mesma nota do modelo oficial?",
             "A2": "Por que este cliente está em risco?",
             "A3": "Que oferta mais reduz o risco de cada cliente?",
             "A4": "Para quem ainda não é cliente, que pacote oferecer?"}
assert set(RESPOSTAS) == set(PERGUNTAS), "falta rodar alguma etapa antes desta célula"
display(pd.DataFrame({"pergunta": PERGUNTAS, "resposta": RESPOSTAS}).rename_axis("#").reset_index())

## Checklist final do 10

| # | Decisão | Status | Onde |
|---|---|---|---|
| D25 | Motor de atendimento | ◐ Nota, motivos e escada de ofertas de cada cliente, e comparação de pacotes para quem ainda não é cliente, com o mesmo cálculo do modelo oficial (conferido na 2.1); relato de cada cliente para separar retenção, recuperação e fidelização e ordenar a lista do atendimento ativo (5.3) (12/11) | Seções 1 a 5; página Atendimento do app |

**Limites a ter em mente:**
- **A queda da nota não é retenção garantida:** é o que o modelo associa à mudança. Medir com um teste (metade da fila
  recebe a oferta, metade não).
- **Cortesia tem custo:** suporte e segurança entram sem mudar a mensalidade na simulação; a empresa decide por quanto
  tempo.
- **O relato é lido por regra de palavras:** serve para escolher a conversa, não entra na nota (D9).
- **Cliente novo no contrato longo é extrapolação:** serve para ordenar os pacotes, não como previsão exata.

**Próximo passo:** publicar o aplicativo (README do `app_monitoramento`); a página Atendimento lê os arquivos gravados
na seção 5.